# Examen de Especialidad — Análisis de la secuencia (YidC)

Notebook para **Google Colab** basado en el Taller de Bioinfo.
Ejecutá las celdas en orden. Los pasos web (BLAST/DeepTMHMM/IUPred/InterPro/
ESPript/AlphaFold/Foldseek) se hacen en el navegador con el setup indicado en
`resultados/entrega_y_setup.md`.

**Menu:** Entorno de ejecución → Ejecutar todas.

## Paso 0 — Instalación y secuencia

In [ ]:
# Instalación (blast+, mafft, clustalo, biopython, pandas, biolib)
!sudo apt-get -qq install ncbi-blast+ mafft clustalo 2> /dev/null 1> /dev/null
!pip -q install biopython pandas pybiolib
print('Listo.')

In [ ]:
# Secuencia del examen -> secuencia.fasta
seq = (
    'MQKILRILIIVAILITTYLLVLAWRDDAANTPKVASATTQAATIDLPNANAGDVPTTTNAS'
    'SDPATTALDGQLISVSTDRYDIRINPVGGDIVHAALKQYDATLNGNEPFVLLESSARTYVA'
    'QSGLIGQDGIDTSAGRATYTSPQMHYEMGENGTLEVPLVYQKDGVTITKTYAFKAGGYPID'
    'LSYNINNASASAWQGQMYAQLKRDDSADPGVEDKGMMGMATYLGGAWGTPDDPYNKLKFGN'
    'FNDGELSVASRDGWVGIVQHYFVSAWTPGDFDAQFYSRNNGNDHFIGFNSPVINVDAGKQI'
    'TMNATLYAGPKVQKELASVAVGLEKTVDYGFLWPISKTLFAVLEVLYKIFGNWGWAIIGLT'
    'ILVKIALFWLSNKSYTSMAKMRAIAPKLQALKDKHGDDRMAMSQEMMQLYRDEKVNPMAGC'
    'LPILIQMPIFLGLYWCLVESVELRHAPWILWIKDLSAMDPWLILPILMTATMFIQQLLNPQ'
    'PADPMQAKMMKIMPLVFAAFMLFFPAGLVLYWTVNNLFSMIHQHWVNKRVEKLT'
)
with open('secuencia.fasta', 'w') as fh:
    fh.write('>query_examen\n' + seq + '\n')
print('Longitud:', len(seq), 'aa')

## Pregunta 1 — Identidad y organismo (BLAST)

Corre `blastp` remoto contra `nr`. En paralelo pegá la secuencia en
https://blast.ncbi.nlm.nih.gov o https://www.uniprot.org/blast .

In [ ]:
# BLASTp remoto (nr). Tarda; si falla la red, usar la web.
!blastp -query secuencia.fasta -db nr -remote -evalue 1e-5 -max_target_seqs 50 \
  -outfmt "6 qseqid sacc pident length qcovs evalue bitscore staxids sscinames stitle" \
  -out blastp_nr.tsv
print('BLAST terminado -> blastp_nr.tsv')

In [ ]:
# Parseo y ranking con pandas
import pandas as pd
cols = ['qseqid','sacc','pident','length','qcovs','evalue','bitscore',
        'staxids','sscinames','stitle']
df = pd.read_table('blastp_nr.tsv', names=cols)
df = df.sort_values('bitscore', ascending=False)
print('BEST HIT:')
print(df.iloc[0][['sacc','pident','qcovs','evalue','sscinames','stitle']])
df[['sacc','pident','qcovs','evalue','bitscore','sscinames','stitle']].head(15)

## Pregunta 2 — Membrana / desorden / dominios

- **Membrana (TM):** celda DeepTMHMM (abajo) o web https://dtu.biolib.com/DeepTMHMM
- **Desorden:** IUPred3 https://iupred.elte.hu/ (long + short, ANCHOR2 on, umbral 0.5)
- **Dominios:** InterPro https://www.ebi.ac.uk/interpro/ y CD-search
  https://www.ncbi.nlm.nih.gov/Structure/cdd/wrpsb.cgi (E=0.01)

In [ ]:
# DeepTMHMM (topología de membrana) via biolib
!biolib run DTU/DeepTMHMM --fasta secuencia.fasta
print('Ver resultados en biolib_results/ (predicted_topologies.3line / .gff3 / plot).')

## Pregunta 3 — Posiciones relevantes (MSA + conservación)

1. Guardá los homólogos del BLAST en `homologos.fasta` (incluir la query).
2. Alineá con **MAFFT L-INS-i** (celda de abajo).
3. Visualizá la conservación en **ESPript** https://espript.ibcp.fr/ESPript/ESPript .

In [ ]:
# Subir/crear homologos.fasta antes de correr esta celda.
import os
if os.path.exists('homologos.fasta'):
    !mafft --localpair --maxiterate 1000 homologos.fasta > msa.aln
    print('MSA -> msa.aln')
else:
    print('Falta homologos.fasta: guardá los homólogos del BLAST y volvé a correr.')

## Pregunta 3/4 — Estructura y comparación

- **Modelo 3D:** AlphaFold3 https://alphafoldserver.com o ColabFold
  (msa_mode=mmseqs2_uniref_env, recycles=3, templates on).
- **Comparación estructural:** Foldseek https://search.foldseek.com/search (PDB100 + AFDB50)
  y RCSB TM-align https://www.rcsb.org/alignment (contra estructuras 'YidC').
- **Validación:** ProSA, PROCHECK (SAVES), QMEANDisCo.

## Pregunta 4 — Integración
Reuní las salidas y redactá la discusión según `resultados/plantilla_informe.md`.